# **VV融合算子开发**
本章将通过代码辅助讲解VV融合算子的基础理论，以及VV融合算子的实现：
- 环境准备
- VV融合算子分析
- VV融合算子工程创建
- VV融合算子实现
- VV融合算子测试
- 课后实践
---

## **1. 环境准备**

本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及msopgen工具。


In [ ]:
!mkdir -p Sources/05.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

安装msopgen工具

In [ ]:
# 下载工具仓代码
!git clone https://gitcode.com/Ascend/msopgen.git
#编译安装msopgen工具
!cd msopgen; python build.py;cd output;pip install mindstudio_opgen*.whl --force-reinstall;pip install mindstudio_opst*.whl --force-reinstall


---
## **2. VV融合算子分析**
假设我们有个简单的小模型，里面包含了一个Add、一个Sub和一个Mul算子，原始的模型如下：  

<img src="./images/vv_model.png">

Add、Sub、Mul 三个算子原生采用串行执行模式，流程拆解为 **9 个独立步骤**：  

- 阶段1（Add）：Add数据搬入 → Add计算 → Add数据搬出  

- 阶段2（Sub）：Sub数据搬入 → Sub计算 → Sub数据搬出  

- 阶段3（Mul）：Mul数据搬入 → Mul计算 → Mul数据搬出

注：Add/Sub/Mul均依赖Vector核运算，因此不改变算法的前提下融合无法减少Vector核的计算耗时，但数据搬运环节存在显著冗余：  

- Add/Sub的输入均为x、y，Mul输入为Add/Sub的输出；  

- 原始流程中x、y需重复参与数据搬入/搬出，算子间中间结果传输无实际价值。

将Add、Sub、Mul融合为一个全新的融合算子后，执行流程简化为 **5 个核心步骤**：
一次性数据搬入（x、y 仅需1次搬入Vector核）→ Add运算 → Sub运算 → Mul运算 → 一次性数据搬出（输出最终结果）。

融合后直接省去 4 个冗余数据传输步骤：  
✅ 省去：Add数据搬出、Sub数据搬入、Sub数据搬出、Mul数据搬入  
✅ 核心收益：大幅降低数据搬运的耗时开销，提升整体执行效率。

---
## **3. VV融合算子IR文件创建**

融合后的算子仅包含输入x、y和输出z，其计算流程为：z = (x + y) * (x - y)。若融合前的算子支持half和float数据类型的ND格式数据运算，则融合后的算子仍兼容这两种数据类型的运算。此处仅以shape为8×2048的输入数据作为演示示例，在真实业务场景下，可灵活调整输入数据的shape参数，也可直接支持泛化输入形式。
<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > SquareDiff</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="3" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">x</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(8, 2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half, float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND, ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">y</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(8, 2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half, float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND, ND</td>
  </tr>
  
  <tr>
  <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">z</td>
    <td style="padding: 8px;">(8, 2048)</td>
    <td style="padding: 8px;">half, float</td>
    <td style="padding: 8px;">ND, ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

我们可以创建一个新的IR文件，命名为vv_fused_op.json，内容如下：


In [ ]:
%%writefile Sources/05.03/vv_fused_op.json
[
    {
        "op": "SquareDiff",
        "input_desc": [
            {
                "name": "x",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["fp16", "float"]
            },
            {
                "name": "y",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["fp16", "float"]
            }
        ],
        "output_desc": [
            {
                "name": "z",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["fp16", "float"]
            }
        ]
    }
]

创建vv_fused_op.json文件后，即可使用msopgen工具生成算子工程，具体步骤如下：

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/05.03/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/05.03/vv_fused_op.json -c ai_core-ascend910b1 -lan cpp -out Sources/05.03/custom_op


命令执行完后，会创建算子工程，目录结构如下所示：
```
custom_op
|--- framework
|--- op_host
|   |--- square_diff.cpp                  // 包含算子原型注册、tiling实现 shape与Dtype推导内容
|   |--- CMakeLists.txt                   // host侧CMakeLists文件，一般不需要修改
|--- op_kernel
|   |--- square_diff_tiling.h             // 算子tiling定义文件   
|   |--- square_diff.cpp                  // 算子代码实现文件 
|   |--- CMakeLists.txt                   // kernel侧CMakeLists文件，一般不需要修改
|--- CMakeLists.txt                       // 根目录CMakeLists文件，一般不需要修改
|--- CMakePresets.json                    // CMake编译配置文件
|--- build.sh                             // 编译脚本
```  
工程目录中的op_kernel和op_host包含了算子的核心实现文件。op_kernel下存放kernel侧算子实现。op_host下存放host侧代码实现，包括算子原型定义、host侧tiling实现。
* **op_host/square_diff.cpp**: 文件名会根据vv_fused_op.json内定义的算子名生成，包含算子原型注册、tiling实现 Shape与Dtype推导内容。
* **op_kernel/square_diff_tiling.h**: 文件名会根据vv_fused_op.json内定义的算子名生成，包含算子tiling结构体定义。
* **op_kernel/square_diff.cpp**: 文件名会根据vv_fused_op.json内定义的算子名生成，包含算子代码实现。
* **CMakePresets.json**: CMake编译配置文件，一般只需要修改ASCEND_CANN_PACKAGE_PATH为实际CANN安装路径。

执行以下命令查看刚刚生成的算子工程目录结构：

In [ ]:
!cd Sources/05.03/custom_op;find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

---
## **4. VV融合算子host侧代码实现**

前文提及，融合后的算子仅包含输入x、y与输出z，其Host侧实现与融合前Add算子基本一致。本文将以固定支持shape为8×2048的输入数据,分到8个核每个核内切分1次处理为例，演示该融合算子的Host侧实现。  
首先在op_kernel/square_diff_tiling.h文件定义tiling结构体，增加totalLength和tileNum字段。

In [ ]:
%%writefile Sources/05.03/custom_op/op_kernel/square_diff_tiling.h
#ifndef SQUARE_DIFF_TILING_H
#define SQUARE_DIFF_TILING_H
#include <cstdint>

struct SquareDiffTilingData {
    uint32_t totalLength;
    uint32_t tileNum;
};
#endif // SQUARE_DIFF_TILING_H

随后在op_host/square_diff.cpp文件Tiling实现函数**TilingFunc**内，对Tiling结构体对象完成赋值操作。

In [ ]:
%%writefile Sources/05.03/custom_op/op_host/square_diff.cpp
#include "../op_kernel/square_diff_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
  SquareDiffTilingData *tiling = context->GetTilingData<SquareDiffTilingData>();
  uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
  tiling->totalLength = totalLength;
  tiling->tileNum = 1;
  context->SetBlockDim(8);
  size_t *currentWorkspace = context->GetWorkspaceSizes(1);
  currentWorkspace[0] = 0;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class SquareDiff : public OpDef {
public:
    explicit SquareDiff(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(SquareDiff);
}

---
## **5. VV融合算子kernel侧代码实现**
完成host侧代码后，即可在op_kernel/square_diff.cpp文件进行kernel侧代码实现。算子类整体写法与之前课程的自定义Add算子基本一致，数据搬入与数据搬出操作不需要变化，仅需修改**Compute函数**内计算逻辑即可。

In [ ]:
%%writefile Sources/05.03/custom_op/op_kernel/square_diff.cpp
#include "kernel_operator.h"
#include "square_diff_tiling.h"
constexpr int32_t BUFFER_NUM = 1; // tensor num for each queue

class KernelSquareDiff {
public:
    __aicore__ inline KernelSquareDiff() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        ascendc_assert(tileNum != 0, "tileNum can not be zero.\n");
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ DTYPE_X *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ DTYPE_Y *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ DTYPE_Z *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(DTYPE_X));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(DTYPE_Y));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(DTYPE_Z));
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<DTYPE_X> xLocal = inQueueX.AllocTensor<DTYPE_X>();
        AscendC::LocalTensor<DTYPE_Y> yLocal = inQueueY.AllocTensor<DTYPE_Y>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<DTYPE_X> xLocal = inQueueX.DeQue<DTYPE_X>();
        AscendC::LocalTensor<DTYPE_Y> yLocal = inQueueY.DeQue<DTYPE_Y>();
        AscendC::LocalTensor<DTYPE_Z> zLocal = outQueueZ.AllocTensor<DTYPE_Z>();
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        AscendC::Sub(xLocal, xLocal, yLocal, this->tileLength);
        AscendC::Mul(zLocal, zLocal, xLocal, this->tileLength);
        outQueueZ.EnQue<DTYPE_Z>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<DTYPE_Z> zLocal = outQueueZ.DeQue<DTYPE_Z>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<DTYPE_X> xGm;
    AscendC::GlobalTensor<DTYPE_Y> yGm;
    AscendC::GlobalTensor<DTYPE_Z> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

extern "C" __global__ __aicore__ void square_diff(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(SquareDiffTilingData);
    GET_TILING_DATA(tilingData, tiling);
    KernelSquareDiff op;
    op.Init(x, y, z, tilingData.totalLength, tilingData.tileNum);
    op.Process();
}

---
## **6. VV融合算子测试**
完成VV融合算子的Host侧与Kernel侧代码后，需要先把算子编译部署。

In [ ]:
!cd Sources/05.03/custom_op;bash build.sh;./build_out/custom_opp_*.run --install-path=${HOME}/

接下来编写测试代码，这里以固定shape为8×2048的half数据，其中x全为2，y全为3进行测试。

In [ ]:
%%writefile Sources/05.03/aclnn_test.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_square_diff.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)
int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));
    const std::vector<int64_t> shape = {8, 2048};
    const int64_t elementCount = shape[0] * shape[1];
    const size_t bufferSize = elementCount * sizeof(aclFloat16);

    void* input0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input0DeviceMem);

    void* input1DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input1DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input1 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input1DeviceMem);

    void* output0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&output0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                         shape.data(), shape.size(), output0DeviceMem);
    std::vector<aclFloat16> input0HostData(elementCount, aclFloatToFloat16(2.0));
    std::vector<aclFloat16> input1HostData(elementCount, aclFloatToFloat16(3.0));
    std::vector<aclFloat16> output0HostData(elementCount, aclFloatToFloat16(0.0));
    std::vector<aclFloat16> goldenData(elementCount, aclFloatToFloat16(-5.0));

    CHECK_ACL(aclrtMemcpy(input0DeviceMem, bufferSize, input0HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(input1DeviceMem, bufferSize, input1HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnSquareDiffGetWorkspaceSize(input0, input1, output0, &workspaceSize, &executor));
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }
    CHECK_ACL(aclnnSquareDiff(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output0HostData.data(), bufferSize, output0DeviceMem,
                          bufferSize, ACL_MEMCPY_DEVICE_TO_HOST));

    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount, 10);
    for (int64_t i = 0; i < previewCount; i++) { printf("%.1f ", aclFloat16ToFloat(output0HostData[i])); }
    printf("\ntest %s\n", std::equal(output0HostData.begin(), output0HostData.end(), goldenData.begin()) ? "pass" : "failed");
    aclDestroyTensor(input0);
    aclDestroyTensor(input1);
    aclDestroyTensor(output0);
    CHECK_ACL(aclrtFree(input0DeviceMem));
    CHECK_ACL(aclrtFree(input1DeviceMem));
    CHECK_ACL(aclrtFree(output0DeviceMem));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    return 0;
}

编译并运行测试代码，检查输出是否与预期一致。

In [ ]:
# 部署算子后编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/05.03/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/05.03/execute_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/05.03/execute_op

执行完成后，会有如下输出：
```
result is:
-5.0 -5.0 -5.0 -5.0 -5.0 -5.0 -5.0 -5.0 -5.0 -5.0 
test pass
```

---
# **课后实践**
上文课程演示了如何实现VV融合算子，并减少数据搬运耗时。现在尝试一下修改Kernel侧实现代码，换种方式实现该融合算子，修改实现方式后可尝试修改代码使算子支持任意Shape输入。  
提示：z = (x + y) * (x - y) = x^2 - y^2

### **host侧代码**  
仅修改实现方式时不需要修改该文件，尝试使算子支持任意Shape输入时需要修改TilingFunc的实现代码。

In [ ]:
%%writefile Sources/05.03/custom_op/op_host/square_diff.cpp
#include "../op_kernel/square_diff_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
  SquareDiffTilingData *tiling = context->GetTilingData<SquareDiffTilingData>();
  uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
  tiling->totalLength = totalLength;
  tiling->tileNum = 1;
  context->SetBlockDim(8);
  size_t *currentWorkspace = context->GetWorkspaceSizes(1);
  currentWorkspace[0] = 0;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class SquareDiff : public OpDef {
public:
    explicit SquareDiff(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(SquareDiff);
}

### **Tiling结构体定义**  
仅修改实现方式时不需要修改该文件，尝试使算子支持任意Shape输入时需要修改Tiling结构体。

In [ ]:
%%writefile Sources/05.03/custom_op/op_kernel/square_diff_tiling.h

#ifndef SQUARE_DIFF_TILING_H
#define SQUARE_DIFF_TILING_H
#include <cstdint>

struct SquareDiffTilingData {
    uint32_t totalLength;
    uint32_t tileNum;
};
#endif // SQUARE_DIFF_TILING_H

### **kernel侧代码**
仅修改实现方式时只需要修改Compute函数内的计算逻辑，尝试使算子支持任意Shape输入时需要修改其他函数。

In [ ]:
%%writefile Sources/05.03/custom_op/op_kernel/square_diff.cpp
#include "kernel_operator.h"
#include "square_diff_tiling.h"
constexpr int32_t BUFFER_NUM = 1; // tensor num for each queue

class KernelSquareDiff {
public:
    __aicore__ inline KernelSquareDiff() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        ascendc_assert(tileNum != 0, "tileNum can not be zero.\n");
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;

        xGm.SetGlobalBuffer((__gm__ DTYPE_X *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ DTYPE_Y *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ DTYPE_Z *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(DTYPE_X));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(DTYPE_Y));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(DTYPE_Z));
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<DTYPE_X> xLocal = inQueueX.AllocTensor<DTYPE_X>();
        AscendC::LocalTensor<DTYPE_Y> yLocal = inQueueY.AllocTensor<DTYPE_Y>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<DTYPE_X> xLocal = inQueueX.DeQue<DTYPE_X>();
        AscendC::LocalTensor<DTYPE_Y> yLocal = inQueueY.DeQue<DTYPE_Y>();
        AscendC::LocalTensor<DTYPE_Z> zLocal = outQueueZ.AllocTensor<DTYPE_Z>();
        // 补充实现代码
        outQueueZ.EnQue<DTYPE_Z>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<DTYPE_Z> zLocal = outQueueZ.DeQue<DTYPE_Z>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<DTYPE_X> xGm;
    AscendC::GlobalTensor<DTYPE_Y> yGm;
    AscendC::GlobalTensor<DTYPE_Z> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

extern "C" __global__ __aicore__ void square_diff(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(SquareDiffTilingData);
    GET_TILING_DATA(tilingData, tiling);
    KernelSquareDiff op;
    op.Init(x, y, z, tilingData.totalLength, tilingData.tileNum);
    op.Process();
}

### **测试代码**
仅修改实现方式时不需要修改该文件，尝试使算子支持任意Shape输入时可以根据自己设计的测试用例修改。

In [ ]:
%%writefile Sources/05.03/aclnn_test.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_square_diff.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)
int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));
    const std::vector<int64_t> shape = {8, 2048};
    const int64_t elementCount = shape[0] * shape[1];
    const size_t bufferSize = elementCount * sizeof(aclFloat16);

    void* input0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input0DeviceMem);

    void* input1DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input1DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input1 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input1DeviceMem);

    void* output0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&output0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                         shape.data(), shape.size(), output0DeviceMem);
    std::vector<aclFloat16> input0HostData(elementCount, aclFloatToFloat16(2.0));
    std::vector<aclFloat16> input1HostData(elementCount, aclFloatToFloat16(3.0));
    std::vector<aclFloat16> output0HostData(elementCount, aclFloatToFloat16(0.0));
    std::vector<aclFloat16> goldenData(elementCount, aclFloatToFloat16(-5.0));

    CHECK_ACL(aclrtMemcpy(input0DeviceMem, bufferSize, input0HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(input1DeviceMem, bufferSize, input1HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnSquareDiffGetWorkspaceSize(input0, input1, output0, &workspaceSize, &executor));
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }
    CHECK_ACL(aclnnSquareDiff(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output0HostData.data(), bufferSize, output0DeviceMem,
                          bufferSize, ACL_MEMCPY_DEVICE_TO_HOST));

    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount, 10);
    for (int64_t i = 0; i < previewCount; i++) { printf("%.1f ", aclFloat16ToFloat(output0HostData[i])); }
    printf("\ntest %s\n", std::equal(output0HostData.begin(), output0HostData.end(), goldenData.begin()) ? "pass" : "failed");
    aclDestroyTensor(input0);
    aclDestroyTensor(input1);
    aclDestroyTensor(output0);
    CHECK_ACL(aclrtFree(input0DeviceMem));
    CHECK_ACL(aclrtFree(input1DeviceMem));
    CHECK_ACL(aclrtFree(output0DeviceMem));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    return 0;
}

修改完成后，重新编译部署算子，运行测试代码，检查输出是否与预期一致。

In [ ]:
# 部署算子
!cd Sources/05.03/custom_op;bash build.sh;./build_out/custom_opp_*.run --install-path=${HOME}/

# 部署算子后编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/05.03/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/05.03/execute_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/05.03/execute_op

**执行以下代码获取答案。**

host实现

In [ ]:
!cat ./answer/05.03_answer/op_host/square_diff.cpp

Tiling结构体定义

In [ ]:
!cat ./answer/05.03_answer/op_kernel/square_diff_tiling.h

Kernel实现

In [ ]:
!cat ./answer/05.03_answer/op_kernel/square_diff.cpp